# Olist Order-to-Cash Performance Analysis
Runs the 8 SQL queries in `02_olist_sql_queries.sql` on the Olist CSV files.

**Definitions** (also at the top of the SQL file): revenue = item prices on orders not canceled or unavailable; late = delivered on a later calendar day than the estimated date; customer = `customer_unique_id`.

In [1]:
import re, os, sqlite3
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 50); pd.set_option('display.max_rows', 100)

## 1. Load the CSV files into a SQLite database
The geolocation file is not used by any query, so it is not loaded.

In [2]:
FILES = {'olist_customers_dataset': 'customers', 'olist_order_items_dataset': 'order_items',
         'olist_order_payments_dataset': 'order_payments', 'olist_order_reviews_dataset': 'order_reviews',
         'olist_orders_dataset': 'orders', 'olist_products_dataset': 'products',
         'olist_sellers_dataset': 'sellers', 'product_category_name_translation': 'category_translation'}

if os.path.exists('olist.db'):
    os.remove('olist.db')
con = sqlite3.connect('olist.db')
rows = []
for f, t in FILES.items():
    df = pd.read_csv(f'data/{f}.csv')
    df.to_sql(t, con, index=False)
    rows.append((t, len(df), int(df.isna().sum().sum()), int(df.duplicated().sum())))
pd.DataFrame(rows, columns=['table', 'rows', 'missing_values', 'duplicate_rows'])

,table,rows,missing_values,duplicate_rows
0,customers,99441,0,0
1,order_items,112650,0,0
2,order_payments,103886,0,0
3,order_reviews,99224,145903,0
4,orders,99441,4908,0
5,products,32951,2448,0
6,sellers,3095,0,0
7,category_translation,71,0,0


## 2. Read the 8 queries from the SQL file
Each query starts after a header line like `-- Q1.` in `02_olist_sql_queries.sql`.

In [4]:
sql = open('02_olist_sql_queries.sql').read()
parts = re.split(r'-- -{20,}\n-- (Q\d)\.', sql)
queries = {}
for i in range(1, len(parts), 2):
    body = parts[i + 1]
    m = re.search(r'\n(WITH|SELECT)', body)
    queries[parts[i]] = body[m.start():].strip().split('\n\n\n')[0].strip()

def run(q):
    return pd.read_sql(queries[q], con)

list(queries)

['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8']

## 3. Q1: Monthly revenue and order trend
Is the business growing? Ignore growth % in the tiny months (2016, Sep-2018).

In [5]:
q1 = run('Q1')
q1

,order_month,orders,revenue,avg_order_value,prev_month_revenue,mom_growth_pct,cumulative_revenue
0,2016-09,2,207.86,103.93,NaN,NaN,207.86
1,2016-10,290,44507.30,153.47,207.86,NaN,44715.16
2,2016-12,1,10.90,10.90,44507.30,-100.0,44726.06
3,2017-01,787,120098.27,152.60,10.90,NaN,164824.33
4,2017-02,1718,244959.35,142.58,120098.27,104.0,409783.68
5,2017-03,2617,368341.32,140.75,244959.35,50.4,778125.00
6,2017-04,2377,353842.98,148.86,368341.32,-3.9,1131967.98
7,2017-05,3640,503159.19,138.23,353842.98,42.2,1635127.17
8,2017-06,3205,429916.61,134.14,503159.19,-14.6,2065043.78
9,2017-07,3946,492287.30,124.76,429916.61,14.5,2557331.08


## 4. Q2: Top 10 product categories by revenue
Which product groups earn the money?

In [6]:
q2 = run('Q2')
q2

,revenue_rank,category,orders,items_sold,revenue,pct_of_total_revenue,cumulative_pct
0,1,health_beauty,8800,9634,1255695.13,9.3,9.3
1,2,watches_gifts,5604,5970,1198185.21,8.9,18.2
2,3,bed_bath_table,9399,11097,1035964.06,7.7,25.9
3,4,sports_leisure,7673,8590,979740.92,7.3,33.2
4,5,computers_accessories,6654,7781,904322.02,6.7,39.9
5,6,furniture_decor,6425,8298,727465.05,5.4,45.3
6,7,housewares,5847,6915,626825.80,4.6,49.9
7,8,cool_stuff,3616,3779,620770.49,4.6,54.5
8,9,auto,3872,4204,586585.73,4.3,58.8
9,10,garden_tools,3505,4328,481009.94,3.6,62.4


## 5. Q3: Delivery time vs estimated date, by customer state
How long do customers wait compared with the date promised?

In [7]:
q3 = run('Q3')
q3

,customer_state,delivered_orders,avg_actual_days,avg_promised_days,avg_days_early_minus_late,slowest_rank
0,RR,41,29.4,46.0,16.6,1
1,AP,67,27.2,46.2,19.1,2
2,AM,145,26.4,45.3,18.9,3
3,AL,397,24.5,32.6,8.0,4
4,PA,946,23.8,37.2,13.4,5
5,MA,717,21.6,30.5,8.9,6
6,SE,335,21.5,30.8,9.3,7
7,CE,1279,21.3,31.4,10.1,8
8,AC,80,21.0,41.1,20.1,9
9,PB,517,20.4,33.0,12.6,10


## 6. Q4: Share of late deliveries by state
Where are deliveries most often late?

In [8]:
q4 = run('Q4')
q4

,customer_state,delivered_orders,late_orders,late_pct,avg_days_late_when_late,worst_rank
0,AL,397,85,21.4,9.5,1
1,MA,717,125,17.4,10.5,2
2,SE,335,51,15.2,16.2,3
3,PI,476,66,13.9,13.3,4
4,CE,1279,176,13.8,15.2,5
5,RR,41,5,12.2,36.4,6
6,BA,3256,396,12.2,12.0,7
7,RJ,12350,1495,12.1,13.5,8
8,PA,946,106,11.2,12.8,9
9,ES,1995,214,10.7,11.3,10


## 7. Q5: Effect of late delivery on review score
How much does lateness lower customer satisfaction?

In [9]:
q5 = run('Q5')
q5

,delivery_bucket,reviewed_orders,avg_review_score,pct_bad_reviews_1_2_stars,pct_five_star
0,1. On time or early,89443,4.29,9.3,62.3
1,2. Late 1-3 days,1852,3.29,32.1,33.2
2,3. Late 4-7 days,1748,2.10,67.7,14.1
3,4. Late more than 7 days,2781,1.70,79.3,7.0


## 8. Q6: Sellers with the highest late / cancellation rates
Sellers with at least 50 orders; top 15 shown.

In [10]:
q6 = run('Q6')
q6

,problem_rank,seller_id,seller_state,orders,late_orders,canceled_orders,late_pct,canceled_pct,problem_pct
0,1,54965bbe3e4f07ae045b90b0b8541f52,PR,78,22,0,28.2,0.0,28.2
1,2,2a1348e9addc1af5aaa619b1a3679d6b,MG,52,13,1,25.0,1.9,26.9
2,3,beadbee30901a7f61d031b6b686095ad,SP,64,15,0,23.4,0.0,23.4
3,4,a49928bcdf77c55c6d6e05e09a9b4ca5,SP,98,21,0,21.4,0.0,21.4
4,5,712e6ed8aa4aa1fa65dab41fed5737e4,SC,79,16,0,20.3,0.0,20.3
5,6,06a2c3af7b3aee5d69171b0e14f0ee87,MA,392,74,0,18.9,0.0,18.9
6,7,b17b679f4f5ce2e03ce6968c62648246,SP,55,8,2,14.5,3.6,18.2
7,8,ea566164622c6b439516ab18062c42cd,SP,50,9,0,18.0,0.0,18.0
8,9,002100f778ceb8431b7a1020ff7ab48f,SP,51,9,0,17.6,0.0,17.6
9,10,88460e8ebdecbfecb5f9601833981930,PR,248,42,1,16.9,0.4,17.3


## 9. Q7: Payment method mix and average order value
Order value here is total payments including freight, so it is higher than the dashboard figure.

In [11]:
q7 = run('Q7')
q7

,payment_type,orders,pct_of_orders,amount_paid,pct_of_amount,avg_order_value,avg_installments
0,credit_card,76505,75.2,12542084.19,78.3,166.47,3.5
1,boleto,19784,19.5,2869361.27,17.9,145.03,1.0
2,voucher,3866,3.8,379436.87,2.4,135.58,1.0
3,debit_card,1528,1.5,217989.79,1.4,142.73,1.0
4,not_defined,3,0.0,0.00,0.0,0.00,1.0


## 10. Q8: Repeat vs one-time customers
Customers are identified by `customer_unique_id`.

In [12]:
q8 = run('Q8')
q8

,segment,customers,pct_of_customers,orders,revenue,pct_of_revenue,avg_revenue_per_customer,avg_orders_per_customer
0,One-time customer,92096,97.0,92096,12743932.31,94.4,138.38,1.00
1,Repeat customer (2+ orders),2887,3.0,6103,750468.43,5.6,259.95,2.11


## 11. Optional: save every result as a CSV

In [14]:
for q in queries:
    run(q).to_csv(f'results/{q}_result.csv', index=False)
print('saved', len(queries), 'files')

saved 8 files
